In [ ]:
import pandas as pd
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

WORKING = Path.cwd().parent / "data_working"
train = pd.read_csv(WORKING / "train.csv", index_col=0)
test  = pd.read_csv(WORKING / "test.csv",  index_col=0)

feature_cols = ["visit_count", "tenure_days", "avg_gap_days",
                "total_spend", "order_count", "age", "sex", "ReaMonths"]
X_train, y_train = train[feature_cols], train["churned"]
X_test,  y_test  = test[feature_cols],  test["churned"]
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

### Train the model

In [ ]:


model2 = XGBClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.03,
    subsample=0.7, colsample_bytree=0.7,
    min_child_weight=5, reg_lambda=2.0,
    eval_metric="auc", random_state=42,
)
model2.fit(X_train, y_train)

tr = roc_auc_score(y_train, model2.predict_proba(X_train)[:, 1])
te = roc_auc_score(y_test,  model2.predict_proba(X_test)[:, 1])
print("Train AUC:", round(tr, 3), "| Test AUC:", round(te, 3))

### Feature Importance

In [ ]:

imp = pd.Series(model2.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp.round(3))

### Save Model

In [ ]:
#save model

import joblib
models_dir = Path.cwd().parent / "models"
models_dir.mkdir(exist_ok=True)
joblib.dump(model2, models_dir / "churn_xgb.joblib")
print("model saved")

### Business Value

In [ ]:
#revenue per cycle 


orders = pd.read_csv(WORKING / "Order_Report.csv")
completed = pd.read_csv(WORKING / "AppointmentExport.csv")
completed = completed[completed["Status"] == "Completed"]

rev_per_cycle = orders["Amount"].sum() / len(completed)
print("revenue per completed visit:", round(rev_per_cycle, 2))

In [ ]:
#business value + lift

test_scored = test.copy()
test_scored["churn_prob"] = model2.predict_proba(X_test)[:, 1]

intervention_success = 0.20
target_frac = 0.20
n_target = int(len(test_scored) * target_frac)
targeted = test_scored.nlargest(n_target, "churn_prob")

model_churners  = targeted["churned"].sum()
base_rate       = test_scored["churned"].mean()
random_churners = base_rate * n_target

model_value  = model_churners  * intervention_success * rev_per_cycle
random_value = random_churners * intervention_success * rev_per_cycle

print(f"Model catches {model_churners} churners vs random {random_churners:.0f}")
print(f"Lift: {model_churners / random_churners:.2f}x")
print(f"Value added by model: £{model_value - random_value:,.0f}")